In [1]:
!pip install pandas matplotlib seaborn -q

import os
import urllib
import urllib.request
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import ast
import json
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

Matplotlib is building the font cache; this may take a moment.


In [2]:
def getKey(keyPath):
    key_dict = {}
    with open(keyPath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                key_dict[k.strip()] = v.strip()
    return key_dict

def buildUrl(key, type_, service, start, end, area_nm=''):
    _url = 'http://openAPI.seoul.go.kr:8088/'
    area_nm_encoded = urllib.parse.quote(area_nm)
    params = '/'.join([key, type_, service, str(start), str(end), '', '', area_nm_encoded])
    return urllib.parse.urljoin(_url, params)

In [3]:
import src.mylib
keyPath = os.path.join(os.getcwd(), 'src', 'key.properties')
key = src.mylib.getKey(keyPath)
KEY = str(key['dataseoul'])
TYPE = 'xml'
SERVICE = 'citydata'
AREA_LIST = [
    '가산디지털단지역', '강남역', '건대입구역', '고덕역', '고속터미널역',
    '교대역', '구로디지털단지역', '구로역', '군자역', '대림역',
    '동대문역', '뚝섬역', '미아사거리역', '발산역', '사당역',
    '삼각지역', '서울대입구역', '서울식물원·마곡나루역', '서울역', '선릉역',
    '성신여대입구역', '수유역', '신논현역·논현역', '신도림역', '신림역',
    '신촌·이대역', '양재역', '역삼역', '연신내역', '오목교역·목동운동장',
    '왕십리역', '용산역', '이태원역', '장지역', '장한평역',
    '천호역', '총신대입구(이수)역', '충정로역', '합정역', '혜화역',
    '홍대입구역(2호선)', '회기역', '쌍문역', '신정네거리역',
    '잠실새내역', '잠실역', '시의회 앞', '숭례문'
]

In [7]:
def fetchAreaData(key, area_nm):
    """단일 지역 혼잡도 데이터 수집 (XML 파싱)"""
    url = buildUrl(key, TYPE, SERVICE, 1, 1, area_nm)
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            xml_data = response.read()
        root = ET.fromstring(xml_data)

        # citydata 노드 탐색
        citydata = root.find('.//CITYDATA')
        if citydata is None:
            return None

        # 인구 혼잡도 노드 탐색
        ppltn = citydata.find('LIVE_PPLTN_STTS/LIVE_PPLTN_STTS')
        if ppltn is None:
            return None

        def getText(node, tag, default=''):
            el = node.find(tag)
            return el.text if el is not None and el.text else default

        # 기본 레코드 생성
        record = {
            'AREA_NM'            : getText(citydata, 'AREA_NM'),
            'AREA_CD'            : getText(citydata, 'AREA_CD'),
            'AREA_CONGEST_LVL'   : getText(ppltn, 'AREA_CONGEST_LVL'),
            'AREA_CONGEST_MSG'   : getText(ppltn, 'AREA_CONGEST_MSG'),
            'AREA_PPLTN_MIN'     : getText(ppltn, 'AREA_PPLTN_MIN'),
            'AREA_PPLTN_MAX'     : getText(ppltn, 'AREA_PPLTN_MAX'),
            'MALE_PPLTN_RATE'    : getText(ppltn, 'MALE_PPLTN_RATE'),
            'FEMALE_PPLTN_RATE'  : getText(ppltn, 'FEMALE_PPLTN_RATE'),
            'PPLTN_RATE_0'       : getText(ppltn, 'PPLTN_RATE_0'),
            'PPLTN_RATE_10'      : getText(ppltn, 'PPLTN_RATE_10'),
            'PPLTN_RATE_20'      : getText(ppltn, 'PPLTN_RATE_20'),
            'PPLTN_RATE_30'      : getText(ppltn, 'PPLTN_RATE_30'),
            'PPLTN_RATE_40'      : getText(ppltn, 'PPLTN_RATE_40'),
            'PPLTN_RATE_50'      : getText(ppltn, 'PPLTN_RATE_50'),
            'PPLTN_RATE_60'      : getText(ppltn, 'PPLTN_RATE_60'),
            'PPLTN_RATE_70'      : getText(ppltn, 'PPLTN_RATE_70'),
            'RESNT_PPLTN_RATE'   : getText(ppltn, 'RESNT_PPLTN_RATE'),
            'NON_RESNT_PPLTN_RATE': getText(ppltn, 'NON_RESNT_PPLTN_RATE'),
            'PPLTN_TIME'         : getText(ppltn, 'PPLTN_TIME'),
        }

        # ✅ 실시간 지하철 승하차 (30분 내)
        sub = citydata.find('LIVE_SUB_PPLTN') # 경로 확인 필요
        if sub is not None:
            record['SUB_30MIN_GTON_MIN']  = getText(sub, 'SUB_30WTHN_GTON_PPLTN_MIN')
            record['SUB_30MIN_GTON_MAX']  = getText(sub, 'SUB_30WTHN_GTON_PPLTN_MAX')
            record['SUB_30MIN_GTOFF_MIN'] = getText(sub, 'SUB_30WTHN_GTOFF_PPLTN_MIN')
            record['SUB_30MIN_GTOFF_MAX'] = getText(sub, 'SUB_30WTHN_GTOFF_PPLTN_MAX')
            record['SUB_STN_TIME']        = getText(sub, 'SUB_STN_TIME')

        # ✅ 실시간 버스 승하차 (30분 내)
        bus = citydata.find('LIVE_BUS_PPLTN') # 경로 확인 필요
        if bus is not None:
            record['BUS_30MIN_GTON_MIN']  = getText(bus, 'BUS_30WTHN_GTON_PPLTN_MIN')
            record['BUS_30MIN_GTON_MAX']  = getText(bus, 'BUS_30WTHN_GTON_PPLTN_MAX')
            record['BUS_30MIN_GTOFF_MIN'] = getText(bus, 'BUS_30WTHN_GTOFF_PPLTN_MIN')
            record['BUS_30MIN_GTOFF_MAX'] = getText(bus, 'BUS_30WTHN_GTOFF_PPLTN_MAX')
            record['BUS_STN_TIME']        = getText(bus, 'BUS_STN_TIME')

        # ✅ 인구 예측 데이터
        fcst_list = []
        for fcst in ppltn.findall('FCST_PPLTN/FCST_PPLTN'):
            fcst_list.append({
                'FCST_TIME'       : getText(fcst, 'FCST_TIME'),
                'FCST_CONGEST_LVL': getText(fcst, 'FCST_CONGEST_LVL'),
                'FCST_PPLTN_MIN'  : getText(fcst, 'FCST_PPLTN_MIN'),
                'FCST_PPLTN_MAX'  : getText(fcst, 'FCST_PPLTN_MAX'),
            })
        record['FCST_PPLTN'] = fcst_list
        
        return record

    except Exception as e:
        print(f'  ⚠️ {area_nm} 수집 실패: {e}')
        return None

In [8]:
rows = []
for area in AREA_LIST:
    ##print(f'  📡 수집 중: {area}')
    result = fetchAreaData(KEY, area)
    if result:
        rows.append(result)
df = pd.DataFrame(rows)
df_station = df.copy()

In [9]:
# 수치형 변환 한 번에
numeric_cols = [
    'AREA_PPLTN_MIN','AREA_PPLTN_MAX',
    'MALE_PPLTN_RATE','FEMALE_PPLTN_RATE',
    'PPLTN_RATE_0','PPLTN_RATE_10','PPLTN_RATE_20','PPLTN_RATE_30',
    'PPLTN_RATE_40','PPLTN_RATE_50','PPLTN_RATE_60','PPLTN_RATE_70',
    'RESNT_PPLTN_RATE','NON_RESNT_PPLTN_RATE',
    'SUB_30MIN_GTON_MIN','SUB_30MIN_GTON_MAX','SUB_30MIN_GTOFF_MIN','SUB_30MIN_GTOFF_MAX',
    'BUS_30MIN_GTON_MIN','BUS_30MIN_GTON_MAX','BUS_30MIN_GTOFF_MIN','BUS_30MIN_GTOFF_MAX'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 평균값 생성
df['PPLTN_AVG']           = (df['AREA_PPLTN_MIN']      + df['AREA_PPLTN_MAX'])      / 2
df['SUB_30MIN_GTON_AVG']  = (df['SUB_30MIN_GTON_MIN']  + df['SUB_30MIN_GTON_MAX'])  / 2
df['SUB_30MIN_GTOFF_AVG'] = (df['SUB_30MIN_GTOFF_MIN'] + df['SUB_30MIN_GTOFF_MAX']) / 2
df['BUS_30MIN_GTON_AVG']  = (df['BUS_30MIN_GTON_MIN']  + df['BUS_30MIN_GTON_MAX'])  / 2
df['BUS_30MIN_GTOFF_AVG'] = (df['BUS_30MIN_GTOFF_MIN'] + df['BUS_30MIN_GTOFF_MAX']) / 2

# 혼잡도 점수
congest_order = {'여유': 1, '보통': 2, '약간 붐빔': 3, '붐빔': 4}
df['CONGEST_SCORE'] = df['AREA_CONGEST_LVL'].map(congest_order)

df[['AREA_NM','AREA_CONGEST_LVL','PPLTN_AVG','SUB_30MIN_GTON_AVG','BUS_30MIN_GTON_AVG']].sort_values('PPLTN_AVG', ascending=False)

,AREA_NM,AREA_CONGEST_LVL,PPLTN_AVG,SUB_30MIN_GTON_AVG,BUS_30MIN_GTON_AVG
29,오목교역·목동운동장,보통,71000.0,375.0,675.0
1,강남역,보통,65000.0,6950.0,1550.0
19,선릉역,보통,41000.0,5150.0,475.0
25,신촌·이대역,보통,35000.0,1650.0,1150.0
2,건대입구역,여유,31000.0,1450.0,275.0
27,역삼역,보통,27000.0,2950.0,375.0
0,가산디지털단지역,보통,21000.0,3850.0,825.0
14,사당역,보통,19000.0,1450.0,475.0
45,잠실역,여유,19000.0,NaN,1250.0
40,홍대입구역(2호선),보통,19000.0,2550.0,925.0


In [10]:
df.to_csv('seoul_station_0413_20.csv', index = False, encoding='utf-8-sig')

In [12]:
import streamlit as st

# UI 구성
theme = st.selectbox("테마", ["데이트", "친구 모임", "가족 나들이"])
age = st.selectbox("연령대", ["10대", "20대", "30대", "40대"])
gender = st.radio("성별", ["여성", "남성", "무관"])
day = st.toggle("주말")

if st.button("장소 추천 받기"):
    result = calcScore(theme, age, gender)
    st.map(result)  # 카카오맵 대신 기본 지도

2026-04-13 20:34:01.961 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 20:34:01.963 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 20:34:01.963 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 20:34:01.965 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 20:34:01.966 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-04-13 20:34:01.968 WARNING streamlit.runtime.scriptrunner_utils.script_run_c